In [11]:
from decimal import Decimal
from html import escape
from typing import Any


def html_escape(arg: Any):
    return escape(str(arg))


def html_int(a: int):
    return f"{a}(<i>{hex(a)!s}</i>)"


def html_real(a: float | Decimal):
    return f"{round(a, 2):.2f}"


def html_str(a: str):
    return html_escape(a).replace("\n", "<br/>\n")


def html_list(l: list[Any] | tuple[Any, ...]):
    items = (f"<li>{htmlize(item)}</li>" for item in l)
    return f"<ul>\n{'\n'.join(items)}\n</ul>"


def html_dict(d: dict[Any, Any]):
    items = (f"<li>{html_escape(k)}={htmlize(v)}</li>" for k, v in d.items())
    return f"<ul>\n{'\n'.join(items)}\n</ul>"


print(
    html_str("""this is
a multi line string
with special characters:  10 < 100""")
)

html_int(255)

html_escape(3 + 10j)

this is<br/>
a multi line string<br/>
with special characters:  10 &lt; 100


'255(<i>0xff</i>)'

'(3+10j)'

In [12]:
from collections.abc import Mapping
from decimal import Decimal
from typing import Any


def htmlize(arg: Any) -> str:
    if isinstance(arg, int):
        return html_int(arg)

    if isinstance(arg, (float, Decimal)):
        return html_real(arg)

    if isinstance(arg, str):
        return html_str(arg)

    if type(arg) in (list, tuple):
        return html_list(arg)

    if type(arg) in (dict, Mapping):
        return html_dict(arg)

    return html_escape(arg)


htmlize(100)
htmlize("""Python
rocks!
""")
print(htmlize([1, 2, 3]))
print(
    htmlize(
        [
            """Python
rocks!
""",
            (10, 20, 30),
            100,
        ]
    )
)

'100(<i>0x64</i>)'

'Python<br/>\nrocks!<br/>\n'

<ul>
<li>1(<i>0x1</i>)</li>
<li>2(<i>0x2</i>)</li>
<li>3(<i>0x3</i>)</li>
</ul>
<ul>
<li>Python<br/>
rocks!<br/>
</li>
<li><ul>
<li>10(<i>0xa</i>)</li>
<li>20(<i>0x14</i>)</li>
<li>30(<i>0x1e</i>)</li>
</ul></li>
<li>100(<i>0x64</i>)</li>
</ul>


In [13]:
from decimal import Decimal
from typing import Any


def htmlize(arg: Any) -> str:
    registry: dict[type, Any] = {
        object: html_escape,
        int: html_int,
        float: html_real,
        Decimal: html_real,
        str: html_str,
        list: html_list,
        tuple: html_list,
        set: html_list,
        dict: html_dict,
    }

    fn = registry.get(type(arg), registry[object])
    return fn(arg)


htmlize(100)
htmlize("""Python
rocks!
""")
print(htmlize([1, 2, 3]))
print(
    htmlize(
        [
            """Python
rocks!
""",
            (10, 20, 30),
            100,
        ]
    )
)

'100(<i>0x64</i>)'

'Python<br/>\nrocks!<br/>\n'

<ul>
<li>1(<i>0x1</i>)</li>
<li>2(<i>0x2</i>)</li>
<li>3(<i>0x3</i>)</li>
</ul>
<ul>
<li>Python<br/>
rocks!<br/>
</li>
<li><ul>
<li>10(<i>0xa</i>)</li>
<li>20(<i>0x14</i>)</li>
<li>30(<i>0x1e</i>)</li>
</ul></li>
<li>100(<i>0x64</i>)</li>
</ul>


In [ ]:
from collections.abc import Callable
from decimal import Decimal
from typing import Any


def single_dispatch(fn: Callable[..., Any]) -> Callable[..., Any]:
    registry = {}

    registry[object] = fn
    registry[int] = lambda a: f"{a}(<i>{hex(a)!s}</i>)"
    registry[str] = lambda s: html_escape(s).replace("\n", "<br/>\n")

    def inner(arg: Any) -> Any:
        return registry.get(type(arg), registry[object])(arg)

    return inner


@single_dispatch
def htmlize(a: Any):
    return escape(str(a))


htmlize("1 < 100")
htmlize(100)

'1 &lt; 100'

'100(<i>0x64</i>)'

In [35]:
from collections.abc import Callable
from typing import Any


def single_dispatch(fn: Callable[..., Any]) -> Callable[..., Any]:
    registry: dict[type, Callable[..., Any]] = {}

    registry[object] = fn

    def decorated(arg: Any) -> Any:
        return registry.get(type(arg), registry[object])(arg)

    def register(type_: Any):
        def decorator(fn: Callable[..., Any]):
            registry[type_] = fn
            return fn

        return decorator

    def dispatch(type_: Any):
        return registry.get(type_, registry[object])

    decorated.register = register
    decorated.registry = registry
    decorated.dispatch = dispatch

    return decorated


@single_dispatch
def htmlize(a: Any):
    return escape(str(a))


htmlize("1 < 100")


@htmlize.register(int)
def html_int(a: Any):
    return f"{a}(<i>{hex(a)!s}</i>)"


htmlize(100)


@htmlize.register(str)
def html_str(s: str):
    return html_escape(s).replace("\n", "<br/>\n")


# html_str = htmlize.register(str)(html_str)


@htmlize.register(tuple)
@htmlize.register(list)
def html_sequence(l: list[Any]):
    items = (f"<li>{htmlize(item)}</li>" for item in l)
    return f"<ul>\n{'\n'.join(items)}\n</ul>"


htmlize([1, 2, 3])
htmlize((1, 2, 3))

htmlize.registry

htmlize.dispatch(int)

'1 &lt; 100'

'100(<i>0x64</i>)'

'<ul>\n<li>1(<i>0x1</i>)</li>\n<li>2(<i>0x2</i>)</li>\n<li>3(<i>0x3</i>)</li>\n</ul>'

'<ul>\n<li>1(<i>0x1</i>)</li>\n<li>2(<i>0x2</i>)</li>\n<li>3(<i>0x3</i>)</li>\n</ul>'

{object: <function __main__.htmlize(a: Any)>,
 int: <function __main__.html_int(a: Any)>,
 str: <function __main__.html_str(s: str)>,
 list: <function __main__.html_sequence(l: list[typing.Any])>,
 tuple: <function __main__.html_sequence(l: list[typing.Any])>}

<function __main__.html_int(a: Any)>

In [51]:
from collections.abc import Sequence
from functools import singledispatch
from numbers import Integral
from typing import Any


@singledispatch
def htmlize(a: Any):
    return escape(str(a))


htmlize.registry
htmlize.dispatch(int)


@htmlize.register(Integral)
def htmlize_integral_number(a: int):
    return f"{a}(<i>{hex(a)!s}</i>)"


htmlize.dispatch(int)
htmlize.dispatch(bool)
isinstance(10, Integral)
isinstance(True, Integral)
htmlize(10)
htmlize(True)


@htmlize.register(Sequence)
def html_sequence(l: list[Any]):
    items = (f"<li>{htmlize(item)}</li>" for item in l)
    return f"<ul>\n{'\n'.join(items)}\n</ul>"


@htmlize.register(str)
def html_string(s: str):
    return html_escape(s).replace("\n", "<br>\n")


@htmlize.register(tuple)
def html_tuple(t: tuple[Any, ...]):
    items = (escape(str(item)) for item in t)
    return f"({', '.join(items)})"


htmlize([1, 2, 3])
htmlize((1, 2, 3))
htmlize("python 1 < 100")

mappingproxy({object: <function __main__.htmlize(a: Any)>})

<function __main__.htmlize(a: Any)>

<function __main__.htmlize_integral_number(a: int)>

<function __main__.htmlize_integral_number(a: int)>

True

...

'10(<i>0xa</i>)'

'True(<i>0x1</i>)'

'<ul>\n<li>1(<i>0x1</i>)</li>\n<li>2(<i>0x2</i>)</li>\n<li>3(<i>0x3</i>)</li>\n</ul>'

'(1, 2, 3)'

'python 1 &lt; 100'

In [57]:
l = [(1, 2), (2, 3), (3, 4)]
for index, (x, y) in enumerate(l):
    print(f"Index: {index}, Coordinate: ({x}, {y})")

Index: 0, Coordinate: (1, 2)
Index: 1, Coordinate: (2, 3)
Index: 2, Coordinate: (3, 4)
